# Recipe Generation Transformer Model Setup

**Purpose**: Setup and configure GPT-2 Medium model for recipe generation from ingredients

**Task**: T018 [P1] [US1] - Setup Recipe Transformer

**Model**: 
- GPT-2 Medium (355M parameters)
- Source: Hugging Face Transformers

**Outputs**:
- Model configuration saved to `models/recipe_generation/`
- Pre-trained GPT-2 model downloaded and cached
- Tokenizer ready for use
- Test generation results

## 1. Environment Setup

In [1]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Check transformers installation
try:
    import torch
    from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config
    print("✅ Transformers and PyTorch installed")
    print(f"   - PyTorch version: {torch.__version__}")
    print(f"   - CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   - CUDA device: {torch.cuda.get_device_name(0)}")
except ImportError as e:
    print("⚠️ Installing required packages...")
    print(f"   Missing: {e}")
    print("\nPlease run: pip install torch transformers")
    raise

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("\n✅ Packages imported successfully")

✅ Transformers and PyTorch installed
   - PyTorch version: 2.7.1+cu118
   - CUDA available: True
   - CUDA device: NVIDIA GeForce RTX 3060 Laptop GPU

✅ Packages imported successfully


## 2. Configure Paths and Directories

In [2]:
# Project directories
PROJECT_ROOT = Path.cwd().parent.parent
MODEL_DIR = PROJECT_ROOT / "models" / "recipe_generation"
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "recipes"
CACHE_DIR = PROJECT_ROOT / "models" / ".cache" / "huggingface"

# Create directories
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
MODEL_NAME = "gpt2-medium"  # 355M parameters
MAX_LENGTH = 512  # Maximum sequence length
TEMPERATURE = 0.9  # Sampling temperature (higher = more creative)
TOP_K = 50  # Top-k sampling
TOP_P = 0.95  # Nucleus sampling
NUM_RETURN_SEQUENCES = 5  # Generate 5 diverse recipes

print(f"📁 Model directory: {MODEL_DIR}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Cache directory: {CACHE_DIR}")
print(f"\n🎯 Model Configuration:")
print(f"   - Model: {MODEL_NAME}")
print(f"   - Max length: {MAX_LENGTH} tokens")
print(f"   - Temperature: {TEMPERATURE}")
print(f"   - Top-k: {TOP_K}")
print(f"   - Top-p: {TOP_P}")
print(f"   - Recipes per ingredient: {NUM_RETURN_SEQUENCES}")

📁 Model directory: c:\Users\Champion\Documents\GitHub\cAIuldron\models\recipe_generation
📁 Data directory: c:\Users\Champion\Documents\GitHub\cAIuldron\data\processed\recipes
📁 Cache directory: c:\Users\Champion\Documents\GitHub\cAIuldron\models\.cache\huggingface

🎯 Model Configuration:
   - Model: gpt2-medium
   - Max length: 512 tokens
   - Temperature: 0.9
   - Top-k: 50
   - Top-p: 0.95
   - Recipes per ingredient: 5


## 3. Download and Load GPT-2 Model

In [3]:
# Set cache directory for Hugging Face
os.environ['TRANSFORMERS_CACHE'] = str(CACHE_DIR)

print("📥 Downloading GPT-2 Medium model...")
print("   (This may take a few minutes on first run)\n")

# Load tokenizer
print("1️⃣ Loading tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=CACHE_DIR
)

# Set padding token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token
print(f"   ✅ Tokenizer loaded")
print(f"   - Vocabulary size: {len(tokenizer)}")
print(f"   - Special tokens: EOS={tokenizer.eos_token}, PAD={tokenizer.pad_token}")

# Load model
print("\n2️⃣ Loading GPT-2 model...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   - Device: {device}")

model = GPT2LMHeadModel.from_pretrained(
    MODEL_NAME,
    cache_dir=CACHE_DIR
)
model.to(device)
model.eval()  # Set to evaluation mode

print(f"   ✅ Model loaded to {device}")
print(f"   - Parameters: {model.num_parameters():,}")
print(f"   - Layers: {model.config.n_layer}")
print(f"   - Hidden size: {model.config.n_embd}")
print(f"   - Attention heads: {model.config.n_head}")

print("\n✅ GPT-2 Medium model ready!")

📥 Downloading GPT-2 Medium model...
   (This may take a few minutes on first run)

1️⃣ Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

   ✅ Tokenizer loaded
   - Vocabulary size: 50257
   - Special tokens: EOS=<|endoftext|>, PAD=<|endoftext|>

2️⃣ Loading GPT-2 model...
   - Device: cuda


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

   ✅ Model loaded to cuda
   - Parameters: 354,823,168
   - Layers: 24
   - Hidden size: 1024
   - Attention heads: 16

✅ GPT-2 Medium model ready!


## 4. Test Text Generation

In [4]:
def generate_text(prompt, max_length=100, temperature=0.9, num_sequences=1):
    """
    Generate text using GPT-2 model
    
    Args:
        prompt (str): Input prompt text
        max_length (int): Maximum generation length
        temperature (float): Sampling temperature (0.0-1.0)
        num_sequences (int): Number of sequences to generate
    
    Returns:
        list: Generated text sequences
    """
    # Encode prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=max_length,
            temperature=temperature,
            top_k=TOP_K,
            top_p=TOP_P,
            num_return_sequences=num_sequences,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=2  # Prevent repetitive phrases
        )
    
    # Decode outputs
    generated_texts = [
        tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]
    
    return generated_texts

# Test generation
test_prompt = "Recipe for chicken breast:"
print(f"🧪 Test prompt: '{test_prompt}'\n")

generated = generate_text(
    test_prompt,
    max_length=150,
    temperature=TEMPERATURE,
    num_sequences=2
)

print("✅ Generation test successful!\n")
for i, text in enumerate(generated, 1):
    print(f"📝 Output {i}:")
    print(f"{text}")
    print("-" * 80)

print("\n💡 Note: The model is pre-trained on general text.")
print("   For better recipe generation, we will fine-tune on recipe datasets.")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


🧪 Test prompt: 'Recipe for chicken breast:'

✅ Generation test successful!

📝 Output 1:
Recipe for chicken breast:

*2-3 lb chicken breasts
 and 2 Tbsp olive oil

--------------------------------------------------------------------------------
📝 Output 2:
Recipe for chicken breast:

Place a chicken thigh in the bottom of a large mixing bowl and whisk together the flour, salt and pepper with an electric mixer for about 5 minutes until all the ingredients are incorporated and light crumbs form on the sides of the bowl.
 Add the chicken pieces and knead for 5-10 minutes or until well mixed. Scrape the mixture into a shallow dish and top with some fresh basil leaves. Drizzle the sauce over the whole chicken and serve with warm bread and fresh parsley.
--------------------------------------------------------------------------------

💡 Note: The model is pre-trained on general text.
   For better recipe generation, we will fine-tune on recipe datasets.


## 5. Save Model Configuration

In [5]:
# Model configuration
config = {
    'model_name': MODEL_NAME,
    'model_type': 'transformer',
    'architecture': 'GPT-2',
    'variant': 'medium',
    'parameters': model.num_parameters(),
    'n_layers': model.config.n_layer,
    'n_heads': model.config.n_head,
    'hidden_size': model.config.n_embd,
    'vocab_size': len(tokenizer),
    'max_length': MAX_LENGTH,
    'inference_params': {
        'temperature': TEMPERATURE,
        'top_k': TOP_K,
        'top_p': TOP_P,
        'num_return_sequences': NUM_RETURN_SEQUENCES,
        'no_repeat_ngram_size': 2
    },
    'device': str(device),
    'cache_dir': str(CACHE_DIR),
    'status': 'pretrained',
    'notes': [
        'Pre-trained GPT-2 Medium from Hugging Face',
        'Will be fine-tuned on recipe datasets for better performance',
        'Generates diverse, creative recipe suggestions',
        'Uses nucleus sampling (top-p) and top-k for quality',
        'Target: Generate 5 unique recipes per ingredient'
    ]
}

config_file = MODEL_DIR / 'model_config.json'
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ Configuration saved to: {config_file}")
print("\n📋 Configuration:")
print(json.dumps(config, indent=2))

✅ Configuration saved to: c:\Users\Champion\Documents\GitHub\cAIuldron\models\recipe_generation\model_config.json

📋 Configuration:
{
  "model_name": "gpt2-medium",
  "model_type": "transformer",
  "architecture": "GPT-2",
  "variant": "medium",
  "parameters": 354823168,
  "n_layers": 24,
  "n_heads": 16,
  "hidden_size": 1024,
  "vocab_size": 50257,
  "max_length": 512,
  "inference_params": {
    "temperature": 0.9,
    "top_k": 50,
    "top_p": 0.95,
    "num_return_sequences": 5,
    "no_repeat_ngram_size": 2
  },
  "device": "cuda",
  "cache_dir": "c:\\Users\\Champion\\Documents\\GitHub\\cAIuldron\\models\\.cache\\huggingface",
  "status": "pretrained",
  "notes": [
    "Pre-trained GPT-2 Medium from Hugging Face",
    "Will be fine-tuned on recipe datasets for better performance",
    "Generates diverse, creative recipe suggestions",
    "Uses nucleus sampling (top-p) and top-k for quality",
    "Target: Generate 5 unique recipes per ingredient"
  ]
}


## 6. Create Helper Functions

In [6]:
def get_model_and_tokenizer(model_dir=CACHE_DIR, device_type='auto'):
    """
    Load GPT-2 model and tokenizer
    
    Args:
        model_dir (Path): Cache directory for model
        device_type (str): 'cuda', 'cpu', or 'auto'
    
    Returns:
        tuple: (model, tokenizer, device)
    """
    # Load tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained(
        MODEL_NAME,
        cache_dir=model_dir
    )
    tokenizer.pad_token = tokenizer.eos_token
    
    # Determine device
    if device_type == 'auto':
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device_type)
    
    # Load model
    model = GPT2LMHeadModel.from_pretrained(
        MODEL_NAME,
        cache_dir=model_dir
    )
    model.to(device)
    model.eval()
    
    return model, tokenizer, device

def estimate_generation_time(num_recipes=5, max_length=512):
    """
    Estimate generation time based on device and parameters
    
    Args:
        num_recipes (int): Number of recipes to generate
        max_length (int): Maximum sequence length
    
    Returns:
        float: Estimated time in seconds
    """
    # Rough estimates (actual time varies by hardware)
    if torch.cuda.is_available():
        time_per_token = 0.01  # ~10ms per token on GPU
    else:
        time_per_token = 0.05  # ~50ms per token on CPU
    
    total_tokens = num_recipes * max_length
    estimated_time = total_tokens * time_per_token
    
    return estimated_time

# Test estimation
est_time = estimate_generation_time(NUM_RETURN_SEQUENCES, MAX_LENGTH)

print("✅ Helper functions defined")
print("\n⏱️ Estimated generation time:")
print(f"   - Device: {device}")
print(f"   - {NUM_RETURN_SEQUENCES} recipes × {MAX_LENGTH} tokens")
print(f"   - Estimated: ~{est_time:.1f} seconds")
print(f"   - Target: < 3 seconds (requires optimization)")

✅ Helper functions defined

⏱️ Estimated generation time:
   - Device: cuda
   - 5 recipes × 512 tokens
   - Estimated: ~25.6 seconds
   - Target: < 3 seconds (requires optimization)


## 7. Model Information Summary

In [7]:
# Calculate model size
param_size_mb = model.num_parameters() * 4 / (1024**2)  # 4 bytes per parameter (fp32)

print("📊 GPT-2 Medium Model Summary")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Size: ~{param_size_mb:.0f} MB (FP32)")
print(f"Layers: {model.config.n_layer}")
print(f"Attention Heads: {model.config.n_head}")
print(f"Hidden Size: {model.config.n_embd}")
print(f"Vocabulary: {len(tokenizer):,} tokens")
print(f"Max Length: {MAX_LENGTH} tokens")
print(f"Device: {device}")
print("=" * 60)

print("\n🎯 Generation Settings:")
print(f"   Temperature: {TEMPERATURE} (creativity)")
print(f"   Top-K: {TOP_K} (diversity)")
print(f"   Top-P: {TOP_P} (nucleus sampling)")
print(f"   Recipes per query: {NUM_RETURN_SEQUENCES}")

print("\n✅ Model ready for recipe generation!")

📊 GPT-2 Medium Model Summary
Model: gpt2-medium
Parameters: 354,823,168
Size: ~1354 MB (FP32)
Layers: 24
Attention Heads: 16
Hidden Size: 1024
Vocabulary: 50,257 tokens
Max Length: 512 tokens
Device: cuda

🎯 Generation Settings:
   Temperature: 0.9 (creativity)
   Top-K: 50 (diversity)
   Top-P: 0.95 (nucleus sampling)
   Recipes per query: 5

✅ Model ready for recipe generation!


## 8. Summary

### ✅ Completed:
1. ✅ Installed transformers and PyTorch
2. ✅ Downloaded GPT-2 Medium model (355M parameters)
3. ✅ Loaded tokenizer and model
4. ✅ Tested text generation
5. ✅ Saved model configuration
6. ✅ Created helper functions

### 🎯 Model Specifications:
- **Model**: GPT-2 Medium
- **Parameters**: ~355 million
- **Architecture**: Transformer (12 layers, 16 heads)
- **Vocabulary**: 50,257 tokens
- **Output**: 5 diverse recipes per ingredient

### 💡 Key Features:
- ✅ **Pre-trained** on large text corpus
- ✅ **Nucleus sampling** for quality generation
- ✅ **Temperature control** for creativity
- ✅ **No repetition** using n-gram blocking

### ⚠️ Current Limitations:
- ⏱️ Generation time may exceed 3s target (needs optimization)
- 📚 Pre-trained on general text (not recipe-specific)
- 🎲 Output quality varies (will improve with fine-tuning)

### 📝 Next Steps:
1. Load recipe training dataset (T019)
2. Fine-tune model on recipe data (optional)
3. Create recipe generation inference notebook (T020)
4. Implement structured output parsing (JSON format)
5. Optimize for <3s generation time
6. Ensure diversity across cuisines

In [8]:
print("🎉 GPT-2 Model Setup Complete!")
print(f"\n📁 Configuration: {config_file}")
print(f"📁 Model cache: {CACHE_DIR}")
print(f"\n✅ Ready for recipe generation pipeline!")

🎉 GPT-2 Model Setup Complete!

📁 Configuration: c:\Users\Champion\Documents\GitHub\cAIuldron\models\recipe_generation\model_config.json
📁 Model cache: c:\Users\Champion\Documents\GitHub\cAIuldron\models\.cache\huggingface

✅ Ready for recipe generation pipeline!
